In [803]:
# Imports and global vars
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('ggplot')
pd.set_option('display.max_columns', 30) # dataset have 21 columns

cwd = Path.cwd()
project_dir = cwd.parent
data_dir = project_dir / 'data'

In [804]:
raw_data = (data_dir / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
# print(raw_data.is_file())
df = pd.read_csv(raw_data)

In [805]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [806]:
row_count, column_count = df.shape
print(f"There are {row_count} rows and {column_count} columns in this dataset")

There are 7043 rows and 21 columns in this dataset


In [807]:
# Guiding Questions for EDA
# 1. What does one row represent? # Answer : Each row represents one customer account. The row includes demographic attributes, subscribed services, account/contract information, billing charges, and the churn label.
# 2. What is the target column? # Answer : the 'Churn' column.
# 3. Is the target balanced or imbalanced? # Answer : The data is imbalanced - 73.4% are not Churn while 26.6% are (in numbers 5174 and 1869).
# 4. Which columns are numeric, categorical, or IDs? # Answer: 
#       customerID is an ID column.
#       tenure, MonthlyCharges are numeric columns.
#       TotalCharges dtype is str but should be numeric. this column stores blank values (' ') and requires cleaning.
#       SeniorCitizen stores integer values but it's categorical.
#       Rest of the columns are categorical.
# 5. Are there missing values or strange data types? # Answer : TotalCharges column stores blank values (' ') and requires investigation why.
# 6. Is there any feature that might not be available at prediction time? # Answer : 
# Churn is the target label and must be excluded during training and prediction. 
# customerID column is an identifier and should be excluded from modeling as well.
# All other columns are assumed to be avialable during prediction

In [808]:
# Answer for Q3 - is the targget value balanced ?
df['Churn'].value_counts(normalize=True)

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [809]:
# Answer for Q4 - numeric vs categorical columns analysis
dtypes = df.dtypes
# print(f'All the dtypes in data frame :\n\n{dtypes}')
print(f"All the columns with non 'str' dtype :\n{dtypes[('str' != dtypes)]}")
print(f"All the columns with 'str' dtype :\n{dtypes[('str' == dtypes)]}")


print(f'\n\nSeniorCitizen have the following values\n{df['SeniorCitizen'].value_counts()}')
print(f'\n\nTotalCharges have the following values\n{df['TotalCharges'].value_counts()}')
print(f'\nMeaning : SeniorCitizen is actually categorical and TotalCharges is numeric')

numeric_columns  = dtypes['str' != dtypes].index.to_list()
category_columns = dtypes['str' == dtypes].index.to_list()

numeric_columns.remove('SeniorCitizen')
numeric_columns.append('TotalCharges')

category_columns.remove('TotalCharges')
category_columns.append('SeniorCitizen')


print(f'\nnumeric columns are {numeric_columns}')
print(f'\ncategorical columns are {category_columns}')

All the columns with non 'str' dtype :
SeniorCitizen       int64
tenure              int64
MonthlyCharges    float64
dtype: object
All the columns with 'str' dtype :
customerID          str
gender              str
Partner             str
Dependents          str
PhoneService        str
MultipleLines       str
InternetService     str
OnlineSecurity      str
OnlineBackup        str
DeviceProtection    str
TechSupport         str
StreamingTV         str
StreamingMovies     str
Contract            str
PaperlessBilling    str
PaymentMethod       str
TotalCharges        str
Churn               str
dtype: object


SeniorCitizen have the following values
SeniorCitizen
0    5901
1    1142
Name: count, dtype: int64


TotalCharges have the following values
TotalCharges
20.2      11
          11
19.75      9
19.9       8
20.05      8
          ..
1990.5     1
7362.9     1
346.45     1
306.6      1
6844.5     1
Name: count, Length: 6531, dtype: int64

Meaning : SeniorCitizen is actually categorical 

In [810]:
# Answer Q5 - missing values, duplicates and strange values

# duplicated rows and or duplicated data for customers
print(df.duplicated().any())
print(df['customerID'].duplicated().any())

False
False


In [811]:
# Missing values
print(df.isna().any(axis=0)) # nan values check
print(df.eq(' ').sum())      # blank values check 

customerID          False
gender              False
SeniorCitizen       False
Partner             False
Dependents          False
tenure              False
PhoneService        False
MultipleLines       False
InternetService     False
OnlineSecurity      False
OnlineBackup        False
DeviceProtection    False
TechSupport         False
StreamingTV         False
StreamingMovies     False
Contract            False
PaperlessBilling    False
PaymentMethod       False
MonthlyCharges      False
TotalCharges        False
Churn               False
dtype: bool
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
Total

In [812]:
# missing values at TotalCharges investigation
# Since MonthlyCharges * tenure =~ TotalCharges
# TotalCharges == 0 --->  tenure == 0 ?
rows_with_blan_tc = df.eq(' ').any(axis=1)
print(df.loc[rows_with_blan_tc, 'tenure'])

488     0
753     0
936     0
1082    0
1340    0
3331    0
3826    0
4380    0
5218    0
6670    0
6754    0
Name: tenure, dtype: int64


In [813]:
## TotalCharges Investigation
# - There are 11 blank values in the TotalCharges. There are no any missing values except those in this dataset.
# - Tenure is 0 For all rows with blank values at TotalCharges.
# - TotalCharges are missing in those instances becuase the tenure is less than a month and no charges were made.
# cleaning options:
#   - Setting TotalCharges to match MonthlyCharges
#   - Convert all blank values to nan and impute them later on
#   - Delete rows entirely
#   - TotalCharges = 0
# - Since MonthlyCharges * tenure ~= TotalCharges (tc is the accumlated charges for a customer for the entire subscription period),
#   0 tenure ---> TotalCharges = 0.
#   Cleaning the data would require to cast entire tc column as float with missing values as 0.

In [814]:
# Cleaning data 
# cleaning TotalCharges column
clean_df = df.copy()
clean_df['TotalCharges']  = pd.to_numeric(clean_df['TotalCharges'], errors='coerce')
clean_df.isna().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [825]:
print(df.dtypes)
df.head(n=10)
df['Partner'].value_counts()
# Data Dictionary 
# | Column | Meaning | Raw dtype | ML role | Notes |
# |---|---|---:|---|---|
# | customerID | Unique customer identifier   | str | exclude | ID column, not useful for generalization |
# | gender | Customer gender (Male or Female) | str | binary categorical | need to inspect |
# | SeniorCitizen | Whether customer is senior citizen | int | binary categorical | Stored as 0/1 |
# | Partner | Whether the customer has a partner or not (Yes, No) | str | binary categorical | need to inspect |
# | Dependents | Whether the customer has dependents or not (Yes, No) | str | binary categorical | need to inspect |
# | tenure | Number of months the customer has stayed with the company | int | numeric feature | Important customer lifecycle variable |
# | PhoneService | Whether the customer has a phone service or not (Yes, No) | str | binary categorical | need to inspect |
# | MultipleLines | Whether the customer has multiple lines or not (Yes, No, No phone service) | str | non binary categorical | need to inspect |
# | InternetService | Customer’s internet service provider (DSL, Fiber optic, No) | str | non binary categorical | need to inspect |
# | OnlineSecurity | Whether the customer has online security or not (Yes, No, No internet service) | str | non binary categorical | need to inspect |
# | OnlineBackup | Whether the customer has online backup or not (Yes, No, No internet service) | str | non binary categorical | need to inspect |
# | DeviceProtection | Whether the customer has device protection or not (Yes, No, No internet service) | str | non binary categorical | need to inspect |
# | TechSupport | Whether the customer has tech support or not (Yes, No, No internet service) | str | non binary categorical | need to inspect |
# | StreamingTV | Whether the customer has streaming TV or not (Yes, No, No internet service) | str | non binary categorical | need to inspect |
# | StreamingMovies | Whether the customer has streaming movies or not (Yes, No, No internet service) | str | non binary categorical | need to inspect |
# | Contract | The contract term of the customer (Month-to-month, One year, Two year) | str | non binary categorical | need to inspect |
# | PaperlessBilling | Whether the customer has paperless billing or not (Yes, No) | str | binary categorical | need to inspect |
# | PaymentMethod | The customer’s payment method (Electronic check, Mailed check, Bank transfer (automatic), Credit card | str | non binary categorical | need to inspect |
# | MonthlyCharges | Monthly bill amount | float | numeric feature | Numeric |
# | TotalCharges | Total accumulated charges | object | numeric feature | Needs conversion; blanks with tenure=0 become 0 |
# | Churn | Whether customer churned | str | target | Binary target: Yes/No |



customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


Partner
No     3641
Yes    3402
Name: count, dtype: int64

In [815]:
# Dataset Questions 
clean_df.fillna(0, inplace=True)

# MonthlyCharges * tenure vs TotalCharges ? 
total = clean_df.shape[0]
equal_index = clean_df.query('TotalCharges == (MonthlyCharges * tenure)').index
bigger_index = clean_df.query('TotalCharges  > (MonthlyCharges * tenure)').index
smaller_index = clean_df.query('TotalCharges < (MonthlyCharges * tenure)').index

num_equal   = clean_df.loc[equal_index].shape[0]
num_bigger  = clean_df.loc[bigger_index].shape[0]
num_smaller = clean_df.loc[smaller_index].shape[0]

print(f"{num_equal}  of {total} customers have their tc equal to (mc * tenure) which are {100 * num_equal / total}%")
print(f"{num_bigger} of {total} customers have their tc bigger from (mc * tenure) which are {100 * num_bigger / total}%")
print(f"{num_smaller} of {total} customers have their tc smaller from (mc * tenure) which are {100 * num_smaller / total}%")




625  of 7043 customers have their tc equal to (mc * tenure) which are 8.874059349708931%
3204 of 7043 customers have their tc bigger from (mc * tenure) which are 45.491977850347865%
3214 of 7043 customers have their tc smaller from (mc * tenure) which are 45.633962799943205%


In [816]:
# MonthlyCharges * tenure < TotalCharges  ? 
print(clean_df.query('MonthlyCharges * tenure < TotalCharges').shape[0])
print(clean_df.query('MonthlyCharges * tenure > TotalCharges').shape[0])

3204
3214


In [817]:
contract_grp = clean_df.groupby('Contract')

# Are MonthlyCharges == TotalCharges if Contract == Month-to-month? 
contract_grp.get_group('Month-to-month') \
            .query('MonthlyCharges == TotalCharges')


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
20,8779-QRDMV,Male,1,No,No,1,No,No phone service,DSL,No,No,Yes,No,No,Yes,Month-to-month,Yes,Electronic check,39.65,39.65,Yes
22,1066-JKSGK,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes
27,8665-UTDHZ,Male,0,Yes,Yes,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,No,Electronic check,30.20,30.20,Yes
33,7310-EGVHZ,Male,0,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Bank transfer (automatic),20.20,20.20,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6979,5351-QESIO,Male,0,No,Yes,1,No,No phone service,DSL,No,No,No,No,No,No,Month-to-month,No,Mailed check,24.20,24.20,No
7010,0723-DRCLG,Female,1,Yes,No,1,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,74.45,74.45,Yes
7016,1471-GIQKQ,Female,0,No,No,1,Yes,No,DSL,No,Yes,No,No,No,No,Month-to-month,No,Electronic check,49.95,49.95,No
7018,1122-JWTJW,Male,0,Yes,Yes,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,70.65,70.65,Yes


In [818]:
# Are MonthlyCharges == TotalCharges / 12 * x-years if Contract == x-years? 
# One year
contract_grp.get_group('One year') \
            .query('MonthlyCharges*12 == TotalCharges')

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


In [819]:
# Two years
contract_grp.get_group('One year') \
            .query('MonthlyCharges*12*2 == TotalCharges')

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


In [820]:
# MonthlyCharges == TotalCharges ? 
clean_df.query('MonthlyCharges != TotalCharges')

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes
